# 2. `ambient=<N>` (node, temperature)

In [1]:
from common import *

shape, log = show(2)

log shape 99: node | temperature | ambient=<N>
54,119 logs (13.2%) from 257 nodes, 2003-12-26 14:58:30 to 2006-04-30 07:01:00
log 274807 at 2006-03-16 16:53:28: 397274 node-77 node temperature 1142528008 1 ambient=26


#### This looks perfectly normal for this specific log, but the verdict relies on the specific temperature.
#### Since the temperature here is just 26°C, it doesn't seem to be an anomaly.
But still, I don't know for sure what should be considered normal for the temperature.
I want to check other logs from this shape - are there ever temperatures that are more extreme?

Every reading of this shape in the 30 days before

In [2]:
# limit=None:  because otherwise search returns only the newest 1000 logs by default
results = pd.DataFrame([asdict(s) for s in logs.search(log.ts - timedelta(days=30), log.ts, log_shape_id=shape.log_shape_id, limit=None)])
print(f"{len(results)} logs from a 30 days range for this shape")
results['temperature'] = results['message'].str.extract(r"ambient=(\d+)").astype(float)
display(results['temperature'].describe())
print(f"Max temperature={results['temperature'].max():.0f}°C.")

5279 logs from a 30 days range for this shape


count    5279.000000
mean       29.278272
std         3.486351
min        23.000000
25%        27.000000
50%        29.000000
75%        32.000000
max        39.000000
Name: temperature, dtype: float64

Max temperature=39°C.


Max=39 celsius, so I'm pretty sure our 26°C is fine. Is there a different shape of logs when the temperature is dangerous?

The log shapes that mention `ambient` or `temperature`

In [3]:
display(find_shapes(text=["ambient", "temperature"]))

,log_shape_id,component,state,template,n_logs,n_nodes,first_seen,last_seen,share
0,99,node,temperature,ambient=<N>,54119,257,2003-12-26 14:58:30,2006-04-30 07:01:00,0.132
1,28,switch_module,temphigh,Temperature (<N>C) exceeds warning threshold,926,10,2005-03-26 09:37:48,2006-02-17 01:17:30,0.002
2,24,node,psu,psu failure\ ambient=<N>,515,91,2004-01-03 08:02:30,2006-04-22 22:22:00,0.001
3,27,node,temphigh,ambient=<N> threshold exceeded,28,14,2004-06-02 16:59:30,2006-02-17 01:06:30,0.000
4,39,node,fan,fan failure\ ambient=<N>,2,1,2004-10-21 20:18:30,2004-10-21 21:56:34,0.000


The `threshold exceeded` logs of the same 30 days

In [4]:
too_hot = as_frame(logs.search(log.ts - timedelta(days=30), log.ts, component="node", state="temphigh", limit=None))
display(too_hot[["ts", "node", "message"]])
too_hot_temperature = too_hot.message.str.extract(r"ambient=(\d+)").astype(float)[0]
print(f"lowest 'threshold exceeded' reading: {too_hot_temperature.min():.0f}, highest plain reading: {results['temperature'].max():.0f}")
assert results['temperature'].max() < too_hot_temperature.min()

,ts,node,message
0,2006-02-17 00:10:00,node-1,ambient=41 threshold exceeded
1,2006-02-17 00:37:00,node-1,ambient=44 threshold exceeded
2,2006-02-17 00:44:30,node-213,ambient=40 threshold exceeded
3,2006-02-17 01:06:30,node-1,ambient=41 threshold exceeded


lowest 'threshold exceeded' reading: 40, highest plain reading: 39


What the node logged around this reading

In [5]:
around_node = timeline(log.node, log.ts - hour, log.ts + 2 * hour)
display(around_node)
assert around_node.iloc[0].template == "boot (command <N>)" and around_node.iloc[0].first_ts < str(log.ts)

,first_ts,last_ts,count,component,state,template
0,2006-03-16 16:47:47,2006-03-16 16:47:47,1,action,start,boot (command <N>)
1,2006-03-16 16:51:31,2006-03-16 16:51:31,1,action,start,wait (command <N>)
2,2006-03-16 16:51:42,2006-03-16 16:51:42,1,unix.hw,net.niff.up,NIFF: node <NODE> has detected an available network connection on network <N>.<N>.<N>.<N> via interface alt<N>
3,2006-03-16 16:51:42,2006-03-16 16:51:42,1,unix.hw,net.niff.up,NIFF: node <NODE> has detected an available network connection on network <N>.<N>.<N>.<N> via interface ee<N>
4,2006-03-16 16:53:28,2006-03-16 16:53:28,1,node,temperature,ambient=<N>
5,2006-03-16 18:21:49,2006-03-16 18:21:49,1,node,status,running


# Verdict: normal, nothing to do. It is probably the temperature a node reports a few minutes after its boot command.
Readings past the threshold came under their own log shape, so a plain 'ambient=<N>' was not an alarm.